# Step 3: Astrometric & Photometric Calibration

After ISR and characterization, pixel values are in instrumental units (counts)
and pixel positions have no sky coordinates. Calibration maps:
- **pixels → sky coordinates** (astrometry)
- **counts → physical flux** (photometry)

**LSST tasks:**
- `lsst.pipe.tasks.calibrate.CalibrateTask` (single-visit)
- `lsst.jointcal.JointcalTask` (multi-visit joint calibration)

**Input:** `icExp` (characterized exposure)  
**Output:** `calexp` (calibrated exposure with WCS + PhotoCalib)

**Reference:** Bosch et al. (2018) §3.5–3.6

## 3.1 Astrometric Calibration (WCS Fitting)

### The problem
The telescope pointing, optical distortions, and atmospheric refraction mean
that the mapping from pixel (x, y) to sky (RA, Dec) is non-trivial. We need
a **World Coordinate System (WCS)** accurate to ~10–50 mas.

### The algorithm
1. **Detect sources** on the characterized exposure
2. **Match** detected sources to a reference catalog (e.g., Gaia DR3, PS1)
   using a pattern-matching algorithm
3. **Fit a WCS model** — a polynomial (SIP) or other distortion model that
   maps (x, y) → (RA, Dec) with residuals << 1 pixel

### Joint calibration (`jointcal`)
For the best astrometry, multiple overlapping visits are solved simultaneously.
This constrains the atmospheric refraction model and optical distortions
self-consistently across the full survey.

### Why it matters for weak lensing
- Coadd images require all visits to be aligned to sub-pixel accuracy
- Positional errors propagate into galaxy-galaxy lensing (source-lens pairing)
- Correlated astrometric errors create spurious shape correlations

```python
# Using the WCS from a calibrated exposure:
wcs = calexp.getWcs()

# Pixel to sky
import lsst.geom as geom
sky_coord = wcs.pixelToSky(geom.Point2D(500, 600))
ra_deg = sky_coord.getRa().asDegrees()
dec_deg = sky_coord.getDec().asDegrees()

# Sky to pixel
pixel_coord = wcs.skyToPixel(sky_coord)
```

## 3.2 Photometric Calibration

### The problem
Raw pixel values are in **ADU** (analog-to-digital units) or electrons.
Different exposures have different zero-points due to airmass, transparency,
and exposure time. We need a calibration that converts to a physical flux unit.

### The algorithm
1. **Measure instrumental magnitudes** of detected stars:
   $m_{\text{inst}} = -2.5 \log_{10}(\text{counts})$
2. **Match** to a photometric reference catalog (PS1, SDSS, SkyMapper)
3. **Fit the zero-point:** $m_{\text{ref}} = m_{\text{inst}} + ZP$
   where ZP absorbs airmass, transparency, and exposure time
4. Store as a `PhotoCalib` object that converts counts → nanojanskys (nJy)

### Joint photometric calibration (`jointcal` / FGCM)
The Forward Global Calibration Method (FGCM) models the atmosphere and
instrument throughput across all visits simultaneously, achieving ~1%
relative photometry across the survey.

### Why it matters for weak lensing
- **Galaxy colours** (needed for photo-z) require consistent photometry
  across bands to ~1% (10 mmag)
- Selection effects from variable depth/zero-point create spurious
  density variations → affect shear correlation functions

```python
# Using PhotoCalib:
photocal = calexp.getPhotoCalib()

# Convert instrumental flux to nJy
flux_nJy = photocal.instFluxToNanojansky(inst_flux, position)

# Convert to AB magnitude
mag_AB = photocal.instFluxToMagnitude(inst_flux, position)

# The zero-point
zp = photocal.getCalibrationMean()  # nJy per count
```

## Flux Units: A Quick Guide

Following Lupton, Gunn & Szalay (1999):

| Unit | Definition | Use |
|------|-----------|-----|
| ADU / counts | Raw detector output | Before calibration |
| nJy (nanojansky) | $10^{-32}$ erg/s/cm²/Hz | LSST standard flux unit |
| AB magnitude | $m_{AB} = -2.5\log_{10}(f/3631\text{ Jy})$ | Standard magnitude system |
| Asinh magnitude | $m = -2.5/\ln(10) \cdot [\text{asinh}(f/(2b f_0)) + \ln(b)]$ | Faint objects ("luptitudes") |

The LSST pipeline stores all calibrated fluxes in **nJy**. The asinh
magnitude system (Lupton et al. 1999) is used for catalog reporting
because it handles zero/negative fluxes gracefully.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Demonstrate the difference between Pogson and asinh magnitudes ---

# Flux in nJy (can be negative due to noise)
flux = np.linspace(-50, 500, 1000)
f0 = 3631e9  # 3631 Jy in nJy (zero-point of AB system)

# Standard (Pogson) magnitude: undefined for flux <= 0
flux_pos = flux[flux > 0]
mag_pogson = -2.5 * np.log10(flux_pos / f0)

# Asinh magnitude (Lupton et al. 1999)
b = 1e-10  # softening parameter (in maggies)
flux_maggies = flux / f0
mag_asinh = -2.5 / np.log(10) * (np.arcsinh(flux_maggies / (2*b)) + np.log(b))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(flux_pos, mag_pogson, 'b-', lw=2, label='Pogson (standard)')
ax.plot(flux, mag_asinh, 'r--', lw=2, label='Asinh ("luptitudes")')
ax.axvline(0, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('Flux (nJy)', fontsize=12)
ax.set_ylabel('Magnitude', fontsize=12)
ax.set_title('Pogson vs. Asinh Magnitudes (Lupton et al. 1999)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(-50, 500)
ax.set_ylim(35, 20)
ax.annotate('Pogson diverges\nat flux=0',
            xy=(5, 32), fontsize=10, color='blue',
            arrowprops=dict(arrowstyle='->', color='blue'),
            xytext=(100, 33))
ax.annotate('Asinh is well-behaved\nfor all fluxes',
            xy=(-20, 30), fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red'),
            xytext=(100, 30))
plt.tight_layout()
plt.savefig('../../figures/pogson_vs_asinh.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Simulate astrometric calibration ---

rng = np.random.default_rng(42)

# "True" positions of reference stars (RA, Dec in degrees)
n_ref = 50
ra_true = 150.0 + rng.uniform(-0.1, 0.1, n_ref)
dec_true = 2.0 + rng.uniform(-0.1, 0.1, n_ref)

# Simulated pixel positions with a simple WCS + distortion
# True WCS: pixel_scale = 0.168 arcsec/pixel, some rotation + distortion
plate_scale = 0.168 / 3600  # degrees/pixel
theta = np.radians(2.5)  # small rotation

# Forward model (sky -> pixel) with 3rd-order distortion
dra = (ra_true - 150.0) / plate_scale
ddec = (dec_true - 2.0) / plate_scale
x_true = (np.cos(theta) * dra - np.sin(theta) * ddec) + 2048
y_true = (np.sin(theta) * dra + np.cos(theta) * ddec) + 2048
# Add cubic distortion
r = np.sqrt((x_true - 2048)**2 + (y_true - 2048)**2)
x_true += 1e-7 * (x_true - 2048) * r**2
y_true += 1e-7 * (y_true - 2048) * r**2

# Measured positions (with noise)
x_meas = x_true + rng.normal(scale=0.3, size=n_ref)  # ~0.3 pixel uncertainty
y_meas = y_true + rng.normal(scale=0.3, size=n_ref)

# Fit a simple linear WCS (ignoring distortion for now)
# x = a0 + a1*dRA + a2*dDec
dra_deg = ra_true - 150.0
ddec_deg = dec_true - 2.0
A = np.column_stack([np.ones(n_ref), dra_deg, ddec_deg])
cx = np.linalg.lstsq(A, x_meas, rcond=None)[0]
cy = np.linalg.lstsq(A, y_meas, rcond=None)[0]

x_fit = A @ cx
y_fit = A @ cy
residual_x = x_meas - x_fit
residual_y = y_meas - y_fit
residual_mas = np.sqrt(residual_x**2 + residual_y**2) * 168  # mas

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quiver plot of astrometric residuals
ax = axes[0]
scale = 50
ax.quiver(x_meas, y_meas, residual_x*scale, residual_y*scale,
          residual_mas, cmap='RdYlBu_r', scale=20)
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title(f'Astrometric residuals (linear fit)\n'
             f'RMS = {np.median(residual_mas):.0f} mas')
ax.set_aspect('equal')

# Residual distribution
ax = axes[1]
ax.hist(residual_mas, bins=15, color='steelblue', edgecolor='white')
ax.axvline(np.median(residual_mas), color='red', ls='--',
           label=f'Median = {np.median(residual_mas):.0f} mas')
ax.set_xlabel('Residual (mas)')
ax.set_ylabel('Count')
ax.set_title('Astrometric residual distribution')
ax.legend()

plt.tight_layout()
plt.savefig('../../figures/astrometric_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print("The linear WCS has systematic residuals from the cubic distortion.")
print("A higher-order (SIP) model would reduce these.")

## The `calexp` — The Calibrated Exposure

After calibration, the exposure is a **`calexp`** — the fundamental data
product for all downstream processing. It contains:

```
calexp
├── image          → pixel values (in counts, calibrated via PhotoCalib)
├── variance       → per-pixel noise variance
├── mask           → quality flags
├── PSF model      → evaluable at any (x,y)
├── WCS            → pixel ↔ sky mapping
├── PhotoCalib     → counts ↔ nJy conversion
├── ApCorrMap      → aperture correction as f(position)
└── metadata       → visit, filter, exposure time, ...
```

This is the input to both **coaddition** (Step 5) and **single-frame
forced photometry** (Step 8).

**Next:** [05_coaddition.ipynb](05_coaddition.ipynb) — Warping and stacking exposures